In [1]:
import os
from datetime import datetime
from urllib.request import urlretrieve

import snowflake.connector
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp, col

In [2]:
YEARS = range(2015, 2026)
MONTHS = range(1, 13)
SERVICES = ["yellow", "green"]

RUN_ID_PREFIX = "run_full_backfill"
RESULTS = []

DOWNLOAD_DIR = "../data/temp_parquet"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [3]:
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("P3 Full RAW Ingest")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4",
            "net.snowflake:snowflake-jdbc:3.13.30"
        ])
    )
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.timestampType", "TIMESTAMP_LTZ")
    .getOrCreate()
)

spark

In [4]:
sfOptions = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": os.getenv("SNOWFLAKE_DATABASE"),
    "sfSchema": os.getenv("SNOWFLAKE_SCHEMA_RAW"),
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

In [5]:
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA_RAW"),
    role=os.getenv("SNOWFLAKE_ROLE"),
)

print("Snowflake connection ready")

Snowflake connection ready


In [6]:
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS YELLOW_TRIPS_RAW (
    vendor_id INTEGER,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count FLOAT,
    trip_distance FLOAT,
    rate_code_id FLOAT,
    store_and_fwd_flag STRING,
    pu_location_id INTEGER,
    do_location_id INTEGER,
    payment_type FLOAT,
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    improvement_surcharge FLOAT,
    total_amount FLOAT,
    congestion_surcharge FLOAT,
    airport_fee FLOAT,
    run_id STRING,
    source_year INTEGER,
    source_month INTEGER,
    service_type STRING,
    ingested_at_utc TIMESTAMP
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS GREEN_TRIPS_RAW (
    vendor_id INTEGER,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    store_and_fwd_flag STRING,
    rate_code_id FLOAT,
    pu_location_id INTEGER,
    do_location_id INTEGER,
    passenger_count FLOAT,
    trip_distance FLOAT,
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    ehail_fee FLOAT,
    improvement_surcharge FLOAT,
    total_amount FLOAT,
    payment_type FLOAT,
    trip_type FLOAT,
    congestion_surcharge FLOAT,
    run_id STRING,
    source_year INTEGER,
    source_month INTEGER,
    service_type STRING,
    ingested_at_utc TIMESTAMP
)
""")

cur.close()
print("RAW tables ready")

RAW tables ready


In [7]:
def get_parquet_url(service_type, year, month):
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/{service_type}_tripdata_{year}-{month:02d}.parquet"


def get_local_parquet_path(service_type, year, month):
    return os.path.join(
        DOWNLOAD_DIR,
        f"{service_type}_tripdata_{year}-{month:02d}.parquet"
    )


def download_parquet_if_needed(service_type, year, month):
    url = get_parquet_url(service_type, year, month)
    local_path = get_local_parquet_path(service_type, year, month)

    if not os.path.exists(local_path):
        print(f"Downloading {service_type} {year}-{month:02d}...")
        urlretrieve(url, local_path)

    return local_path


def read_parquet_spark(service_type, year, month):
    local_path = download_parquet_if_needed(service_type, year, month)
    return spark.read.parquet(local_path)

In [8]:
def transform_yellow(df, year, month, run_id):
    df_std = (
        df.withColumnRenamed("VendorID", "vendor_id")
          .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
          .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
          .withColumnRenamed("RatecodeID", "rate_code_id")
          .withColumnRenamed("PULocationID", "pu_location_id")
          .withColumnRenamed("DOLocationID", "do_location_id")
          .withColumnRenamed("Airport_fee", "airport_fee")
          .withColumn("pickup_datetime", col("pickup_datetime").cast("timestamp"))
          .withColumn("dropoff_datetime", col("dropoff_datetime").cast("timestamp"))
          .withColumn("payment_type", col("payment_type").cast("double"))
          .withColumn("run_id", lit(run_id))
          .withColumn("source_year", lit(year))
          .withColumn("source_month", lit(month))
          .withColumn("service_type", lit("yellow"))
          .withColumn("ingested_at_utc", current_timestamp().cast("timestamp"))
          .select(
              "vendor_id",
              "pickup_datetime",
              "dropoff_datetime",
              "passenger_count",
              "trip_distance",
              "rate_code_id",
              "store_and_fwd_flag",
              "pu_location_id",
              "do_location_id",
              "payment_type",
              "fare_amount",
              "extra",
              "mta_tax",
              "tip_amount",
              "tolls_amount",
              "improvement_surcharge",
              "total_amount",
              "congestion_surcharge",
              "airport_fee",
              "run_id",
              "source_year",
              "source_month",
              "service_type",
              "ingested_at_utc"
          )
    )
    return df_std

In [9]:
def transform_green(df, year, month, run_id):
    df_std = (
        df.withColumnRenamed("VendorID", "vendor_id")
          .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
          .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
          .withColumnRenamed("RatecodeID", "rate_code_id")
          .withColumnRenamed("PULocationID", "pu_location_id")
          .withColumnRenamed("DOLocationID", "do_location_id")
          .withColumn("pickup_datetime", col("pickup_datetime").cast("timestamp"))
          .withColumn("dropoff_datetime", col("dropoff_datetime").cast("timestamp"))
          .withColumn("payment_type", col("payment_type").cast("double"))
          .withColumn("run_id", lit(run_id))
          .withColumn("source_year", lit(year))
          .withColumn("source_month", lit(month))
          .withColumn("service_type", lit("green"))
          .withColumn("ingested_at_utc", current_timestamp().cast("timestamp"))
          .select(
              "vendor_id",
              "pickup_datetime",
              "dropoff_datetime",
              "store_and_fwd_flag",
              "rate_code_id",
              "pu_location_id",
              "do_location_id",
              "passenger_count",
              "trip_distance",
              "fare_amount",
              "extra",
              "mta_tax",
              "tip_amount",
              "tolls_amount",
              "ehail_fee",
              "improvement_surcharge",
              "total_amount",
              "payment_type",
              "trip_type",
              "congestion_surcharge",
              "run_id",
              "source_year",
              "source_month",
              "service_type",
              "ingested_at_utc"
          )
    )
    return df_std

In [10]:
def get_existing_count(target_table, service_type, year, month):
    cur = conn.cursor()
    try:
        cur.execute(f"""
            SELECT COUNT(*)
            FROM {target_table}
            WHERE service_type = '{service_type}'
              AND source_year = {year}
              AND source_month = {month}
        """)
        return cur.fetchone()[0]
    finally:
        cur.close()

In [11]:
def delete_existing_month(target_table, service_type, year, month):
    cur = conn.cursor()
    try:
        cur.execute(f"""
            DELETE FROM {target_table}
            WHERE service_type = '{service_type}'
              AND source_year = {year}
              AND source_month = {month}
        """)
    finally:
        cur.close()

In [12]:
def write_spark_df_to_snowflake(df, target_table):
    (
        df.write
          .format("snowflake")
          .options(**sfOptions)
          .option("dbtable", target_table)
          .option("column_mapping", "name")
          .mode("append")
          .save()
    )

In [13]:
def ingest_one_month(service_type, year, month):
    run_id = f"{RUN_ID_PREFIX}_{service_type}_{year}_{month:02d}"
    print(f"Starting {service_type} {year}-{month:02d}")

    try:
        df = read_parquet_spark(service_type, year, month)

        if service_type == "yellow":
            df_std = transform_yellow(df, year, month, run_id)
            target_table = "YELLOW_TRIPS_RAW"
        else:
            df_std = transform_green(df, year, month, run_id)
            target_table = "GREEN_TRIPS_RAW"

        expected_rows = df_std.count()
        existing_rows = get_existing_count(target_table, service_type, year, month)

        if expected_rows == 0:
            RESULTS.append({
                "service_type": service_type,
                "year": year,
                "month": month,
                "status": "failed",
                "rows_written": 0,
                "existing_rows": existing_rows,
                "expected_rows": expected_rows,
                "message": "empty dataframe"
            })
            print(f"Failed {service_type} {year}-{month:02d}: empty dataframe")
            return

        if existing_rows == expected_rows and expected_rows > 0:
            RESULTS.append({
                "service_type": service_type,
                "year": year,
                "month": month,
                "status": "skipped_existing",
                "rows_written": 0,
                "existing_rows": existing_rows,
                "expected_rows": expected_rows,
                "message": "already loaded"
            })
            print(f"Skipped {service_type} {year}-{month:02d}: already loaded ({existing_rows} rows)")
            return

        if existing_rows > 0:
            print(f"Replacing existing data for {service_type} {year}-{month:02d} ({existing_rows} rows found)")
            delete_existing_month(target_table, service_type, year, month)

        write_spark_df_to_snowflake(df_std, target_table)

        RESULTS.append({
            "service_type": service_type,
            "year": year,
            "month": month,
            "status": "success",
            "rows_written": expected_rows,
            "existing_rows": existing_rows,
            "expected_rows": expected_rows,
            "message": "ok"
        })

        print(f"Done {service_type} {year}-{month:02d} -> {expected_rows} rows")

    except Exception as e:
        RESULTS.append({
            "service_type": service_type,
            "year": year,
            "month": month,
            "status": "failed",
            "rows_written": 0,
            "existing_rows": None,
            "expected_rows": None,
            "message": str(e)
        })
        print(f"Failed {service_type} {year}-{month:02d}: {e}")

In [14]:
ingest_one_month("yellow", 2024, 1)
ingest_one_month("green", 2024, 1)

Starting yellow 2024-01
Done yellow 2024-01 -> 2964624 rows
Starting green 2024-01
Done green 2024-01 -> 56551 rows


In [15]:
for service_type in SERVICES:
    for year in YEARS:
        for month in MONTHS:
            ingest_one_month(service_type, year, month)

Starting yellow 2015-01
Done yellow 2015-01 -> 12741035 rows
Starting yellow 2015-02
Done yellow 2015-02 -> 12442394 rows
Starting yellow 2015-03
Done yellow 2015-03 -> 13342951 rows
Starting yellow 2015-04
Done yellow 2015-04 -> 13063758 rows
Starting yellow 2015-05
Done yellow 2015-05 -> 13157677 rows
Starting yellow 2015-06
Done yellow 2015-06 -> 12324936 rows
Starting yellow 2015-07
Done yellow 2015-07 -> 11559666 rows
Starting yellow 2015-08
Done yellow 2015-08 -> 11123123 rows
Starting yellow 2015-09
Done yellow 2015-09 -> 11218122 rows
Starting yellow 2015-10
Done yellow 2015-10 -> 12307333 rows
Starting yellow 2015-11
Done yellow 2015-11 -> 11305240 rows
Starting yellow 2015-12
Done yellow 2015-12 -> 11452996 rows
Starting yellow 2016-01
Done yellow 2016-01 -> 10905067 rows
Starting yellow 2016-02
Done yellow 2016-02 -> 11375412 rows
Starting yellow 2016-03
Done yellow 2016-03 -> 12203824 rows
Starting yellow 2016-04
Done yellow 2016-04 -> 11927996 rows
Starting yellow 2016-05


In [16]:
import pandas as pd

results_df = pd.DataFrame(RESULTS)
results_df

,service_type,year,month,status,rows_written,existing_rows,expected_rows,message
0,yellow,2024,1,success,2964624,0,2964624,ok
1,green,2024,1,success,56551,0,56551,ok
2,yellow,2015,1,success,12741035,0,12741035,ok
3,yellow,2015,2,success,12442394,0,12442394,ok
4,yellow,2015,3,success,13342951,0,13342951,ok
...,...,...,...,...,...,...,...,...
261,green,2025,8,success,46306,0,46306,ok
262,green,2025,9,success,48893,0,48893,ok
263,green,2025,10,success,49416,0,49416,ok
264,green,2025,11,success,46912,0,46912,ok


In [17]:
results_df.groupby(["service_type", "status"]).size()

service_type  status          
green         skipped_existing      1
              success             132
yellow        skipped_existing      1
              success             132
dtype: int64

In [18]:
results_df.to_csv("ingestion_results_full_range.csv", index=False)
print("Saved results to ingestion_results_full_range.csv")

Saved results to ingestion_results_full_range.csv


In [19]:
conn.close()
spark.stop()
print("Snowflake connection closed and Spark session stopped")

Snowflake connection closed and Spark session stopped
